<a href="https://colab.research.google.com/github/dev-jotape/favela_environmental_trajectories/blob/main/scripts/02_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
setwd("/content/drive/MyDrive/Yale_Environmental_Data_Science_2026-2027/favela_environmental_trajectories")

library(dplyr)
library(tidyr)
library(readr)
library(foreign)   # reads the shapefile attribute table (.dbf) without sf

## 2. Settings

In [69]:
RAW_PIXELS  <- "data/raw/mapbiomas_c11_favela_pixels_pilot.csv"
FAVELA_DBF  <- "data/raw/state_of_brazilian_favelas.dbf"

OUT_DIR     <- "data/processed"
OUT_PIXELS  <- file.path(OUT_DIR, "2_favela_pixels_clean_pilot.csv")
OUT_FAVELAS <- file.path(OUT_DIR, "2_favela_urbanization_pilot.csv")
OUT_LOOKUP  <- file.path(OUT_DIR, "2_class_lookup.csv")
OUT_DICT    <- file.path(OUT_DIR, "2_data_dictionary.csv")

PILOT_MUN   <- 3550308   # São Paulo (IBGE municipality code)
URBAN_CLASS <- 24        # MapBiomas "4.2. Urban Area"
FIRST_YEAR  <- 1985      # first year of the series: urban here means "1985 or earlier"

## 3. Read the raw pixels

In [54]:
pixels_raw <- read_csv(RAW_PIXELS,
                       col_types = cols(CD_FCU = col_character(), .default = col_double()))
favelas_raw <- read.dbf(FAVELA_DBF, as.is = TRUE)

dim(pixels_raw)
head(pixels_raw)

dim(favelas_raw)
head(favelas_raw)

[1] 77264    44

CD_FCU,longitude,latitude,classification_1985,classification_1986,classification_1987,classification_1988,classification_1989,classification_1990,classification_1991,⋯,classification_2016,classification_2017,classification_2018,classification_2019,classification_2020,classification_2021,classification_2022,classification_2023,classification_2024,classification_2025
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
35503080069,-46.37207,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24
35503080069,-46.37180,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24
35503080069,-46.37153,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24
35503080069,-46.37126,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24
35503080069,-46.37099,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24
35503080069,-46.37072,-23.49925,24,24,24,24,24,24,24,⋯,24,24,24,24,24,24,24,24,24,24


[1] 12344    88

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,distance,dist_km,dist_norm,⋯,prim_hosp,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population,area_km2,density_km
,<dbl>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
1,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,23646.847,23.646847,0.53315855,⋯,752.1925,2071.7241,0.9974501,0.4510280,0.6812075,0.3215357,1,1159,0.026,45062.21
2,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,2544.012,2.544012,0.08035357,⋯,281.6608,1167.1932,1.0000000,0.9788084,0.8729724,0.9998000,0,340,0.117,2916.00
3,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,28454.701,28.454701,0.50765095,⋯,0.0000,1003.2633,1.0000000,1.0000000,0.8536706,0.9998000,0,394,0.028,14189.51
4,33045570104,Parque Nossa Senhora da Penha,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,1286.545,1.286545,0.01880914,⋯,683.3772,659.1353,0.9907505,0.8929206,0.5796455,1.0000000,0,1058,0.016,65726.53
5,35095020135,Núcleo Residencial Jardim das Andorinhas II,35,São Paulo,SP,3509502,Campinas,5245.591,5.245591,0.14682443,⋯,1383.1634,1488.4671,0.9910004,0.6996963,0.8280753,0.9998000,1,658,0.037,17788.59
6,33033020044,Morro do Peixe Galo,33,Rio de Janeiro,RJ,3303302,Niterói,6209.613,6.209613,0.54898627,⋯,320.4103,1134.3215,0.9925004,0.9151168,0.8773060,0.9998000,0,856,0.079,10824.89


## 4. Tidy the column names and key types

In [14]:
# Pixels: lowercase names, and shorten "classification_1985" to "y1985"
pixels <- pixels_raw %>%
  rename_with(tolower) %>%
  rename_with(function(x) sub("classification_", "y", x), starts_with("classification_"))

# Favelas: lowercase names, keep only the pilot municipality,
# make the key text (same type as in the pixel file) and label the regimes
favelas <- favelas_raw %>%
  rename_with(tolower) %>%
  filter(cd_mun == PILOT_MUN) %>%
  mutate(cd_fcu = format(cd_fcu, scientific = FALSE, trim = TRUE),
         regime = factor(cluster, levels = 0:2,
                         labels = c("Structured", "Partially Structured", "Unstructured"))) %>%
  select(
    cd_fcu, nm_fcu, cd_mun, nm_mun, sigla_uf, regime,
    area_km2, population, wtr_idx, sanit_idx,
    slope_10, slope_20, slope_30, slope_30p,
    hand_0_3, hand_3_6, hand_6p
  )

# Check unique cd_fcu key
favelas %>% count(cd_fcu) %>% filter(n > 1)

[1] "cd_fcu"    "longitude" "latitude"  "y1985"     "y1986"

[1] 1357

cd_fcu,n
<chr>,<int>


## 5. Missing values and duplicated pixels

In [22]:
year_cols <- grep("^y\\d{4}$", names(pixels), value = TRUE)

# How many missing values per year, and how many duplicated pixels?
colSums(is.na(pixels[year_cols]))[colSums(is.na(pixels[year_cols])) > 0]
sum(duplicated(pixels[c("cd_fcu", "longitude", "latitude")]))

n_before <- nrow(pixels)

pixels <- pixels %>%
  distinct(cd_fcu, longitude, latitude, .keep_all = TRUE) %>%   # drop duplicated pixels
  filter(if_all(all_of(year_cols), function(x) !is.na(x))) %>%   # drop incomplete trajectories
  mutate(pixel_id = row_number())                                # one ID per pixel

n_before          # rows before
nrow(pixels)      # rows after

named numeric(0)

[1] 0

[1] 77264

[1] 77264

## ~6. Reshape wide → long with `pivot_longer()`~

In [24]:
# Kept wide (one column per year) to keep the file small;

# pixels_long <- pixels %>%
#   pivot_longer(
#     cols            = all_of(year_cols),   # the 41 year columns to stack
#     names_to        = "year",              # old column names go into "year"
#     names_prefix    = "y",                 # "y1985" -> "1985"
#     names_transform = list(year = as.integer),
#     values_to       = "class"              # land cover codes go into "class"
#   )

# nrow(pixels)        # pixels
# nrow(pixels_long)   # pixels x 41 years

[1] 77264

[1] 3167824

## 7. Give the class codes a meaning with `left_join()`

In [57]:
# Year columns as a matrix: one row per pixel, one column per year
class_mat <- as.matrix(pixels[year_cols])
years     <- as.integer(sub("y", "", year_cols))

# Lookup table: translates MapBiomas class codes into broad land cover groups
# https://brasil.mapbiomas.org/wp-content/uploads/sites/3/2026/08/CodigosDeLegenda_LegendCodes_MapBiomas_Brazil_Collection11_PDF.pdf
class_lookup <- tribble(
  ~class, ~group,
  # 1. Forest
  1,  "forest",               # Forest
  3,  "forest",               # Forest Formation
  6,  "forest",               # Floodable Forest
  4,  "forest",               # Savanna Formation
  7,  "forest",               # Flooded Savanna (beta)
  5,  "forest",               # Mangrove
  49, "forest",               # Wooded Sandbank Vegetation
  # 2. Herbaceous or shrubby vegetation
  10, "herbaceous_shrubby",   # Herbaceous or Shrubby Vegetation
  12, "herbaceous_shrubby",   # Grassland Formation
  77, "herbaceous_shrubby",   # Herbaceous and Shrub Formation
  11, "herbaceous_shrubby",   # Wetland
  84, "herbaceous_shrubby",   # Salt Marsh (beta)
  50, "herbaceous_shrubby",   # Herbaceous Sandbank Vegetation
  32, "herbaceous_shrubby",   # Hypersaline Tidal Flat
  29, "herbaceous_shrubby",   # Rocky Outcrop
  # 3. Farming
  15, "pasture",              # Pasture
  14, "agriculture",          # Farming
  18, "agriculture",          # Agriculture
  19, "agriculture",          # Temporary Crop
  39, "agriculture",          # Soybean
  20, "agriculture",          # Sugar cane
  40, "agriculture",          # Rice
  62, "agriculture",          # Cotton (beta)
  41, "agriculture",          # Other Temporary Crops
  36, "agriculture",          # Perennial Crop
  46, "agriculture",          # Coffee
  47, "agriculture",          # Citrus
  35, "agriculture",          # Palm Oil
  48, "agriculture",          # Other Perennial Crops
  9,  "agriculture",          # Forest Plantation
  21, "mosaic_of_uses",       # Mosaic of Uses
  # 4. Non vegetated area
  24, "urban",                # Urban Area
  22, "other_non_vegetated",  # Non vegetated area
  23, "other_non_vegetated",  # Beach, Dune and Sand Spot
  30, "other_non_vegetated",  # Mining
  75, "other_non_vegetated",  # Photovoltaic Power Plant
  91, "other_non_vegetated",  # Wind farm (beta)
  25, "other_non_vegetated",  # Other non Vegetated Areas
  # 5. Water
  26, "water",                # Water
  33, "water",                # River, Lake and Ocean
  31, "water"                 # Aquaculture
)

# Check: codes present in the data but missing from the lookup (review these)
setdiff(unique(as.vector(class_mat)), class_lookup$class)

numeric(0)

## 8. Check classification noise ("flicker")
How often does a pixel go from **urban back to non-urban**?

In [52]:
# Year columns as a matrix: one row per pixel, one column per year
class_mat <- as.matrix(pixels[year_cols])
urban_mat <- class_mat == URBAN_CLASS   # TRUE where the pixel is urban in that year

# A reversal = urban in one year (columns 1..40) and not urban the next (columns 2..41)
flicker <- tibble(
  pixel_id    = pixels$pixel_id,
  n_reversals = rowSums(urban_mat[, -ncol(urban_mat)] & !urban_mat[, -1])
)

# How many pixels have 0, 1, 2... reversals (%)
flicker %>%
  count(n_reversals) %>%
  mutate(pct = round(100 * n / sum(n), 2))


PERSIST <- 5

n_reversals,n,pct
<dbl>,<int>,<dbl>
0,75492,97.71
1,1497,1.94
2,221,0.29
3,35,0.05
4,14,0.02
5,5,0.01


## 9. First persistent urban year and previous land cover (per pixel)

**Persistence rule:** a pixel counts as urbanized in year *y* only if it is urban in *y* and in the next `PERSIST − 1` years.

- `first_year`: first persistent urban year (`NA` = never urbanized)
- `censored`: urban already in 1985, so the real year is 1985 or earlier
- `prev_class`, `prev_group`: land cover in the year **before** urbanization (`NA` if censored or never urban)

In [45]:
# Persistent = urban this year AND in each of the next PERSIST-1 years
# (years after 2025 do not exist, so they count as TRUE)
persistent <- urban_mat
for (k in seq_len(PERSIST - 1)) {
  shifted    <- cbind(urban_mat[, -(1:k), drop = FALSE],     # shift k years to the left
                      matrix(TRUE, nrow(urban_mat), k))      # fill the missing future years
  persistent <- persistent & shifted
}

has_urban <- rowSums(persistent) > 0                         # pixel became urban at some point
first_idx <- max.col(persistent, ties.method = "first")      # column of the first persistent year
years     <- as.integer(sub("y", "", year_cols))

pixel_urb <- tibble(
  pixel_id   = pixels$pixel_id,
  first_year = ifelse(has_urban, years[first_idx], NA_integer_),
  # class in the year before urbanization (none if already urban in 1985)
  prev_class = ifelse(has_urban & first_idx > 1,
                      class_mat[cbind(seq_len(nrow(class_mat)), pmax(first_idx - 1, 1))],
                      NA_real_)
) %>%
  left_join(class_lookup, by = c("prev_class" = "class")) %>%   # class code -> group
  rename(prev_group = group) %>%
  mutate(censored = !is.na(first_year) & first_year == FIRST_YEAR)

# Share of pixels in each situation (%)
pixel_urb %>%
  summarise(never_urban          = mean(is.na(first_year)),
            urban_in_1985        = mean(censored),
            urbanized_after_1985 = mean(!is.na(first_year) & !censored)) %>%
  mutate(across(everything(), function(x) round(100 * x, 1)))

never_urban,urban_in_1985,urbanized_after_1985
<dbl>,<dbl>,<dbl>
5,45.7,49.4


## 10. Build the clean **pixel-level** table

In [64]:
pixels_clean <- pixels %>%
  left_join(pixel_urb, by = "pixel_id") %>%                     # first urban year, previous cover
  left_join(flicker,   by = "pixel_id") %>%                     # number of urban -> non-urban reversals
  left_join(favelas %>% select(cd_fcu, regime), by = "cd_fcu") %>%   # infrastructural regime
  relocate(pixel_id, cd_fcu, regime, longitude, latitude,       # identifiers first,
           first_year, censored, prev_class, prev_group, n_reversals)  # then derived variables

# Check 1: row count must not change
nrow(pixels) == nrow(pixels_clean)


[1] TRUE

## 11. Aggregate to the **favela level** with `group_by()` + `summarise()`

In [48]:
favela_metrics <- pixels_clean %>%
  group_by(cd_fcu) %>%
  summarise(
    n_pixels          = n(),
    share_never_urban = mean(is.na(first_year)),
    urb_year_p25      = quantile(first_year, 0.25, type = 1, na.rm = TRUE),
    urb_year_median   = quantile(first_year, 0.50, type = 1, na.rm = TRUE),
    urb_year_p75      = quantile(first_year, 0.75, type = 1, na.rm = TRUE),
    urban_share_1985  = mean(y1985 == URBAN_CLASS),
    urban_share_2025  = mean(y2025 == URBAN_CLASS),
    mean_reversals    = mean(n_reversals)
  ) %>%
  mutate(urb_span        = urb_year_p75 - urb_year_p25,
         median_censored = urb_year_median == FIRST_YEAR)

# Previous land cover: share of each group among pixels urbanized after 1985.
# count() gives one row per favela and group -> pivot_wider() spreads groups into columns
favela_prev <- pixels_clean %>%
  filter(!is.na(prev_group)) %>%
  count(cd_fcu, prev_group) %>%
  group_by(cd_fcu) %>%
  mutate(n_converted = sum(n), share = n / n_converted) %>%
  ungroup() %>%
  select(-n) %>%
  pivot_wider(names_from = prev_group, values_from = share,
              names_prefix = "prev_", values_fill = 0)

nrow(favela_metrics)   # favelas with at least one pixel

[1] 1357

## 12. Combine with the favela attributes and check the join


In [67]:
favelas_clean <- favelas %>%
  left_join(favela_metrics, by = "cd_fcu") %>%
  left_join(favela_prev,    by = "cd_fcu")

# Check: row count must not change (one row per favela)
nrow(favelas) == nrow(favelas_clean)

[1] TRUE

## 13. Write clean data

In [70]:
data_dictionary <- tribble(
  ~file,            ~variable,               ~description,
  "both",           "cd_fcu",                "IBGE favela code (2022 Census), text",
  "both",           "regime",                "Infrastructural regime from da Silva et al. (2026): Structured, Partially Structured, Unstructured",
  "pixels",         "pixel_id",              "Pixel identifier (row number after cleaning)",
  "pixels",         "longitude, latitude",   "Pixel center, WGS84",
  "pixels",         "y1985 ... y2025",       "MapBiomas Collection 11 class code in each year (see 2_class_lookup.csv)",
  "pixels",         "first_year",            "First year urban (class 24) and still urban for PERSIST consecutive years; NA = never",
  "pixels",         "censored",              "TRUE if already urban in 1985 (true year is 1985 or earlier)",
  "pixels",         "prev_class, prev_group", "Land cover in the year before the pixel became urban",
  "pixels",         "n_reversals",           "Times the pixel switched from urban to non-urban (classification noise)",
  "favelas",        "nm_fcu, cd_mun, nm_mun, sigla_uf", "Favela name, municipality code and name, state",
  "favelas",        "area_km2, population",  "Favela area (km2) and population (2022 Census)",
  "favelas",        "wtr_idx, sanit_idx",    "Improved water and sanitation indices (0-1) from the preprint",
  "favelas",        "hand_0_3",              "Share of area within 3 m of the nearest drainage (HAND)",
  "favelas",        "slope_30p",             "Share of area with slope above 30%",
  "favelas",        "n_pixels",              "Number of 30 m pixels whose center falls inside the favela",
  "favelas",        "share_never_urban",     "Share of pixels never persistently urban (1985-2025)",
  "favelas",        "urb_year_p25 / median / p75", "Percentiles of first_year across the favela's pixels (censored pixels enter as 1985)",
  "favelas",        "urb_span",              "urb_year_p75 - urb_year_p25: how spread out urbanization was (years)",
  "favelas",        "median_censored",       "TRUE if the median year is 1985 (mostly urban before the series)",
  "favelas",        "urban_share_1985, urban_share_2025", "Share of pixels classified as urban in 1985 and 2025",
  "favelas",        "mean_reversals",        "Average n_reversals across the favela's pixels",
  "favelas",        "n_converted",           "Pixels urbanized after 1985 (base for prev_* shares)",
  "favelas",        "prev_<group>",          "Share of pixels urbanized after 1985 whose previous land cover was <group>"
)

stopifnot(dir.exists(OUT_DIR))
write_csv(pixels_clean,    OUT_PIXELS)
write_csv(favelas_clean,   OUT_FAVELAS)
write_csv(class_lookup,    OUT_LOOKUP)
write_csv(data_dictionary, OUT_DICT)

list.files(OUT_DIR)   # check that the files are there

[1] "2_class_lookup.csv"              "2_data_dictionary.csv"          
[3] "2_favela_pixels_clean_pilot.csv" "2_favela_urbanization_pilot.csv"